# VoiceStudio on Google Colab (GPU)
Generate high-quality Indonesian TTS (OmniVoice, 600+ languages) on a free Colab GPU.
Model cache persists on **Google Drive** — download once, reuse later.

### How to use
1. File → New Notebook → paste/import this notebook (or open from GitHub).
2. Runtime → Change runtime type → **T4 GPU**.
3. Run cells top-to-bottom. First run downloads the model (~2.5GB) into your Drive; later runs just mount it.
4. In the last cell, paste your narration text → run → `audio_out.mp3` is saved to your Drive.
5. The printed **tunnel URL + API key** let any local pipeline (e.g. Hermes) call the `/generate` endpoint remotely.


## 1. Mount Google Drive
Mount your Drive so the model cache can be saved between sessions.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Point model cache to Drive
Caches are stored under `MyDrive/omnivoice_colab/hf`. Only the first session downloads ~2.5GB.


In [ ]:
import os
DRIVE = '/content/drive/MyDrive'
OMNI_DIR = os.path.join(DRIVE, 'omnivoice_colab')
os.makedirs(OMNI_DIR, exist_ok=True)
os.environ['HF_HOME'] = os.path.join(OMNI_DIR, 'hf')
os.environ['HF_HUB_CACHE'] = os.path.join(OMNI_DIR, 'hf', 'hub')
print('Cache dir:', os.environ['HF_HOME'])


## 3. Install tools
Installs Cloudflared (tunnel) and VoiceStudio (clone + pip install).


In [ ]:
# --- Cloudflared (tunnel) ---
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

# --- VoiceStudio source (skip jika sudah ada) ---
import os
if not os.path.exists('/content/VoiceStudio/backend/main.py'):
    !git clone --depth 1 https://github.com/debpalash/VoiceStudio /content/VoiceStudio

# --- Python 3.11 env (deps VoiceStudio butuh <3.13; Colab default >=3.13) ---
if not os.path.exists('/opt/conda/envs/vs/bin/python'):
    !wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O /tmp/mf.sh
    !bash /tmp/mf.sh -b -p /opt/conda
    !/opt/conda/bin/conda create -y -n vs python=3.11
!/opt/conda/envs/vs/bin/pip install -q -e /content/VoiceStudio
print('install done, python:', '/opt/conda/envs/vs/bin/python')


## 4. Start TTS server
Creates an API key, saves it to Drive (`api_key.txt`) and starts VoiceStudio on port 3900.


In [ ]:
import os, subprocess, time, secrets, requests
# Generate a stable API key (reused across sessions so the pipeline key stays the same)
KEY_FILE = os.path.join(OMNI_DIR, 'api_key.txt')
KEY = open(KEY_FILE).read().strip() if os.path.exists(KEY_FILE) else 'vsc_' + secrets.token_hex(16)
open(KEY_FILE, 'w').write(KEY)
print('API key:', KEY)

env = dict(os.environ)
env['OMNIVOICE_API_KEY'] = KEY
env['OMNIVOICE_SERVER_MODE'] = '1'
env['OMNIVOICE_ANALYTICS_DISABLED'] = '1'
log = open(os.path.join(OMNI_DIR, 'server.log'), 'a')
PY311 = '/opt/conda/envs/vs/bin/python'
server = subprocess.Popen(
    [PY311, '-m', 'uvicorn', 'backend.main:app', '--host', '0.0.0.0', '--port', '3900'],
    cwd='/content/VoiceStudio', env=env, stdout=log, stderr=subprocess.STDOUT)

# Wait until the TTS engine is actually ready (503 = still loading/downloading model)
deadline = time.time() + 900   # first run downloads ~2.5GB; allow up to 15 min
url = 'http://127.0.0.1:3900'
while time.time() < deadline:
    try:
        r = requests.get(url + '/engines/tts', headers={'Authorization': 'Bearer ' + KEY}, timeout=5)
        if r.status_code == 200 and r.json().get('active', r.json().get('active_model')):
            print('Server ready on :3900')
            break
        print('warming up...', r.status_code, flush=True)
    except Exception as e:
        print('warming up...', repr(e), flush=True)
    time.sleep(5)
else:
    print('--- server.log tail ---')
    print(open(os.path.join(OMNI_DIR, 'server.log')).read()[-3000:])
    raise RuntimeError('Server did not become ready in time')


## 5. Open a public tunnel
Exposes the server to the internet so your local pipeline can reach it.


In [ ]:
import re, time, subprocess
tunnel = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://localhost:3900'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
TUNNEL_URL = None
for line in iter(tunnel.stdout.readline, ''):
    m = re.search(r'(https://[a-z0-9-]+\.trycloudflare\.com)', line)
    if m:
        TUNNEL_URL = m.group(1)
        break
if not TUNNEL_URL:
    raise RuntimeError('Could not get tunnel URL')
print('Tunnel URL:', TUNNEL_URL)
print('Endpoints: {}/generate'.format(TUNNEL_URL))


## 6. Generate audio
Paste your narration into `TEXT` and run. The MP3 is saved to your Drive. Long text is **auto-chunked** and stitched.


In [ ]:
import requests, re, subprocess

def _chunk(text, max_words=28):
    sentences = [s.strip() for s in re.findall(r'[^.!?\n]+[.!?]?', text) if s.strip()]
    chunks, cur, wc = [], [], 0
    for s in sentences:
        n = len(s.split())
        if cur and wc + n > max_words:
            chunks.append(' '.join(cur)); cur, wc = [], 0
        cur.append(s); wc += n
    if cur:
        chunks.append(' '.join(cur))
    return chunks or [text]

def generate(text, instruct='female, young adult, high pitch', language='id'):
    parts = []
    for chunk in _chunk(text):
        r = requests.post(TUNNEL_URL + '/generate',
                          files={'text': (None, chunk), 'language': (None, language),
                                 'instruct': (None, instruct)},
                          headers={'Authorization': 'Bearer ' + KEY}, timeout=600)
        r.raise_for_status()
        parts.append(r.content)
    if len(parts) == 1:
        return parts[0]
    # stitch chunks then encode to MP3
    for i, p in enumerate(parts):
        open(f'/tmp/chunk_{i}.mp3', 'wb').write(p)
    inp, fc = [], []
    for i in range(len(parts)):
        inp += ['-i', f'/tmp/chunk_{i}.mp3']; fc.append(f'[{i}:a]')
    fc.append(f'concat=n={len(parts)}:v=0:a=1[out]')
    subprocess.run(['ffmpeg', '-y', '-v', 'error'] + inp +
                   ['-filter_complex', ''.join(fc), '-map', '[out]',
                    '-c:a', 'libmp3lame', '-b:a', '128k', '/tmp/stitched.mp3'], check=True)
    return open('/tmp/stitched.mp3', 'rb').read()

# --- KONFIGURASI ---
TEXT = ""  # <-- tempel narasi di sini
OUTPUT = os.path.join(OMNI_DIR, 'audio_out.mp3')

if TEXT.strip():
    audio = generate(TEXT)
    open(OUTPUT, 'wb').write(audio)
    print('Saved:', OUTPUT, f'({len(audio)} bytes)')
else:
    print('Isi TEXT dulu, lalu run ulang cell ini.')
